In [1]:
import os
import pandas as pd

from sqlalchemy import create_engine, text, URL
from langchain_core.tools import tool

# Read credentials from environment variables
db_url = URL.create(
    drivername="postgresql+psycopg",
    username=os.getenv("POSTGRES_USER", "postgres"),
    password=os.getenv("POSTGRES_PASSWORD"),
    host="localhost",
    port=5432,
    database="ecommerce_analytics"
)

engine = create_engine(db_url)

# Test the connection
with engine.connect() as connection:
    print(connection.execute(text("SELECT 1")).scalar())

print("PostgreSQL connected successfully!")


1
PostgreSQL connected successfully!


In [2]:

# from previous notebook
def detect_anomalies(threshold=100, limit=10):
    query = text("""
        SELECT
            "InvoiceNo",
            "Description",
            "Quantity",
            "Revenue",
            CASE
                WHEN "Quantity" > :high_threshold THEN 'High Risk'
                WHEN "Quantity" > :threshold THEN 'Review'
                ELSE 'Normal'
            END AS anomaly_status
        FROM online_retail
        WHERE "Quantity" > :threshold
        ORDER BY "Quantity" DESC
        LIMIT :limit;
    """)

    with engine.connect() as connection:
        results = pd.read_sql(
            query,
            connection,
            params={
                "threshold": threshold,
                "high_threshold": threshold * 10,
                "limit": limit
            }
        )

    return results

def get_monthly_revenue():
    query = text("""
        WITH monthly_sales AS (
            SELECT
                "Month",
                SUM("Revenue") AS monthly_revenue
            FROM online_retail
            GROUP BY "Month"
        ),
        monthly_comparison AS (
            SELECT
                "Month",
                monthly_revenue,
                LAG(monthly_revenue) OVER (
                    ORDER BY "Month"
                ) AS previous_revenue
            FROM monthly_sales
        )
        SELECT
            "Month",
            ROUND(monthly_revenue::numeric, 2) AS revenue,
            ROUND(previous_revenue::numeric, 2) AS previous_revenue,
            ROUND(
                (
                    (monthly_revenue - previous_revenue)
                    / NULLIF(previous_revenue, 0) * 100
                )::numeric,
                2
            ) AS growth_percentage
        FROM monthly_comparison
        ORDER BY "Month";
    """)

    with engine.connect() as connection:
        results = pd.read_sql(query, connection)

    return results

def get_top_products(limit=10):
    query = text("""
        SELECT
            "StockCode",
            "Description",
            SUM("Quantity") AS total_units_sold,
            ROUND(SUM("Revenue")::numeric, 2) AS total_revenue,
            COUNT(DISTINCT "InvoiceNo") AS total_invoices
        FROM online_retail
        WHERE "Quantity" > 0
          AND "Revenue" > 0
          AND "Description" IS NOT NULL
        GROUP BY "StockCode", "Description"
        ORDER BY total_revenue DESC
        LIMIT :limit;
    """)

    with engine.connect() as connection:
        results = pd.read_sql(
            query,
            connection,
            params={"limit": limit}
        )

    return results


In [3]:
# now converting functions to langchain tools
@tool
def anomaly_detection_tool(threshold:int = 100, limit:int = 10):
    """Find unusually high-quantity transactions.

    Use for questions about suspicious transactions or sales anomalies.
    Threshold sets the minimum quantity to flag.
    Limit sets the maximum quantity to flag.
    """
    res = detect_anomalies(threshold=threshold, limit=limit)
    return res.to_json(orient='records')

@tool
def monthly_revenue_tool() -> str:
    """
    Analyze monthly revenue and month over month growth.

    Use for questions about monthly sales sales, revenue trends, revenue increases, or revenue declines.
    """
    res = get_monthly_revenue()
    return res.to_json(orient='records')#records/columns/index

@tool
def top_products_tool(limit: int = 10)->str:
    """
    Find products ranked by total revenue.
    Use for questions about top products, product performance  or highest product revenue.
    """
    res = get_top_products(limit=limit)
    return res.to_json(orient='records')

In [4]:

# Test 1: Top 5 products
print("TOP PRODUCTS")
print(top_products_tool.invoke({"limit": 5}))

# Test 2: Monthly revenue
print("\nMONTHLY REVENUE")
print(monthly_revenue_tool.invoke({}))

# Test 3: Anomalies above quantity 500
print("\nANOMALIES")
print(anomaly_detection_tool.invoke({
    "threshold": 500,
    "limit": 5
}))


TOP PRODUCTS
[{"StockCode":"DOT","Description":"DOTCOM POSTAGE","total_units_sold":706.0,"total_revenue":206248.77,"total_invoices":706},{"StockCode":"22423","Description":"REGENCY CAKESTAND 3 TIER","total_units_sold":13879.0,"total_revenue":174484.74,"total_invoices":1988},{"StockCode":"23843","Description":"PAPER CRAFT , LITTLE BIRDIE","total_units_sold":80995.0,"total_revenue":168469.6,"total_invoices":1},{"StockCode":"85123A","Description":"WHITE HANGING HEART T-LIGHT HOLDER","total_units_sold":37599.0,"total_revenue":104340.29,"total_invoices":2189},{"StockCode":"47566","Description":"PARTY BUNTING","total_units_sold":18295.0,"total_revenue":99504.33,"total_invoices":1685}]

MONTHLY REVENUE
[{"Month":"2010-12","revenue":823746.14,"previous_revenue":null,"growth_percentage":null},{"Month":"2011-01","revenue":691364.5600000001,"previous_revenue":823746.14,"growth_percentage":-16.07},{"Month":"2011-02","revenue":523631.89,"previous_revenue":691364.5600000001,"growth_percentage":-24.2

In [8]:

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

print("LLM initialized successfully!")

LLM initialized successfully!


In [1]:
tools = [
    anomaly_detection_tool,
    monthly_revenue_tool,
    top_products_tool
]

print("Registered tools:")
for t in tools:
    print("-", t.name)
llm_with_tools = llm.bind_tools(tools)

print("LLM connected to analytics tools!")

NameError: name 'llm' is not defined